In [0]:
import re
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

base_path = "s3://bucket-retail-sales/raw_data/"

def clean_column_names(df):
    def to_snake_case(name):
        cleaned = re.sub(r'[^a-zA-Z0-9]', '', name.strip())
        s1 = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', cleaned)
        snake = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1)
        return re.sub(r'_+', '_', snake).lower()
    return df.toDF(*[to_snake_case(c) for c in df.columns])

files = {
    "crm_cust_base": "CRM_CUST_BASE_RAW.csv",
    "inv_levels":    "INV_LEVELS_RAW.csv",
    "product":       "MST_PRODUCT_MASTER.csv",
    "sales":         "RAW_SALES_DTL.csv",
    "stores_geo":    "STORES_GEO_LIST_FINAL.csv",
    "rtn_trans":     "RTN_TRANS_LOGS.csv",
}

for table_name, filename in files.items():
    df = spark.read \
        .format("csv") \
        .option("header", "true") \
        .option("inferSchema", "false") \
        .load(base_path + filename)
    
    df = clean_column_names(df)
    
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"bronze.{table_name}")